### Imports go here

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, roc_auc_score, log_loss, brier_score_loss
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier

### Load the data

In [ ]:
men_df = pd.read_csv("../data/m_tournament_training_dataset.csv")

### Get familiar with the data, just in case

In [88]:
men_df.columns

Index(['Season', 'Team1ID', 'Team2ID', 'Target', 'WinPctDiff', 'SeedNumDiff',
       'NetRatingDiff', 'OffEffDiff', 'DefEffDiff', 'MarginDiff',
       'ReboundPctDiff', 'TurnoverPctDiff', 'FGPctDiff', 'ThreePctDiff',
       'FTPctDiff', 'RankingDiff'],
      dtype='str')

In [ ]:
men_df.info()

### Function to calculate the metrics

In [ ]:
def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5, print_results=True):
    # Predicted probabilities for the positive class
    y_prob = model.predict_proba(X_test)[:, 1]

    # Convert probabilities into class predictions using threshold
    y_pred = (y_prob >= threshold).astype(int)

    # Standard classification metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    # Confusion matrix and report
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred)

    # Probability-based metrics
    logloss = log_loss(y_test, y_prob)
    brier = brier_score_loss(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    results = {
        "accuracy": acc,
        "f1_score": f1,
        "precision": precision,
        "recall": recall,
        "classification_report": report,
        "confusion_matrix": cm,
        "log_loss": logloss,
        "brier_score": brier,
        "auc": auc
    }

    if print_results:
        print("=== Classification Metrics ===")
        print(f"Accuracy:   {acc:.4f}")
        print(f"F1 Score:   {f1:.4f}")
        print(f"Precision:  {precision:.4f}")
        print(f"Recall:     {recall:.4f}")

        print("\n=== Classification Report ===")
        print(report)

        print("=== Confusion Matrix ===")
        print(cm)

        print("\n=== Probability Metrics ===")
        print(f"Log Loss:   {logloss:.4f}")
        print(f"Brier Score:{brier:.4f}")
        print(f"AUC:        {auc:.4f}")

    return results

### Our data has already been taken of in previous notebooks, so we just need to split the data before starting to model

In [ ]:
X = men_df.drop("Target", axis=1)
y = men_df["Target"]

In [89]:
X.columns

Index(['Season', 'Team1ID', 'Team2ID', 'WinPctDiff', 'SeedNumDiff',
       'NetRatingDiff', 'OffEffDiff', 'DefEffDiff', 'MarginDiff',
       'ReboundPctDiff', 'TurnoverPctDiff', 'FGPctDiff', 'ThreePctDiff',
       'FTPctDiff', 'RankingDiff'],
      dtype='str')

In [91]:
y

0       1
1       1
2       1
3       1
4       1
       ..
2893    0
2894    0
2895    0
2896    0
2897    0
Name: Target, Length: 2898, dtype: int64

### Let's split the data by using Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Models

Source: https://scikit-learn.org/stable/supervised_learning.html

- Classification Task
    - Logistic Regression
    - Lasso
    - Ridge
    - SVM
    - Decision Tree
    - Random Forest
    - Voting Classifier
    - XGBoost / AdaBoost
    - MLP Classifier (https://scikit-learn.org/stable/modules/neural_networks_supervised.html#classification)

### Logistic Regression

In [ ]:
logistic_model = LogisticRegression(max_iter=500, random_state=42)
logistic_model.fit(X_train, y_train)

In [ ]:
y_pred = logistic_model.predict(X_test)

### Metrics
- Making a funciton to calculate all the metrics necessary for each model to improve readability

### Accuracy
- The percentage of predictions the model got correct overall.
### Precision
- Of the cases the model predicted as positive, how many were actually positive.
### Recall
- Of the cases that were actually positive, how many the model correctly found.
### F1 Score
- A balance between precision and recall, useful when you care about both types of mistakes.

### Confusion Matrix
- TP → predicted positive and actually positive
- FN → predicted negative but actually positive
- FP → predicted positive but actually negative
- TN → predicted negative and actually negative

### Most common metrics for binary competition
- Log Loss
- Brier Score
- AUC

In [ ]:
results = evaluate_binary_classifier(logistic_model, X_test, y_test)

### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
decisionT_model = DecisionTreeClassifier()

In [ ]:
decisionT_model.fit(X_train, y_train)

In [ ]:
y_pred = decisionT_model.predict(X_test)

In [83]:
results = evaluate_binary_classifier(decisionT_model, X_test, y_test)

=== Classification Metrics ===
Accuracy:   0.5948
F1 Score:   0.5884
Precision:  0.6022
Recall:     0.5753

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.59      0.61      0.60       288
           1       0.60      0.58      0.59       292

    accuracy                           0.59       580
   macro avg       0.60      0.59      0.59       580
weighted avg       0.60      0.59      0.59       580

=== Confusion Matrix ===
[[177 111]
 [124 168]]

=== Probability Metrics ===
Log Loss:   14.6039
Brier Score:0.4052
AUC:        0.5950


### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
random_forest_model = RandomForestClassifier(max_depth=5,  random_state=42)

In [ ]:
random_forest_model.fit(X_train, y_train)

In [ ]:
y_pred = random_forest_model.predict(X_test)

In [82]:
results = evaluate_binary_classifier(random_forest_model, X_test, y_test)

=== Classification Metrics ===
Accuracy:   0.7241
F1 Score:   0.7232
Precision:  0.7308
Recall:     0.7158

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.72      0.73      0.73       288
           1       0.73      0.72      0.72       292

    accuracy                           0.72       580
   macro avg       0.72      0.72      0.72       580
weighted avg       0.72      0.72      0.72       580

=== Confusion Matrix ===
[[211  77]
 [ 83 209]]

=== Probability Metrics ===
Log Loss:   0.5548
Brier Score:0.1875
AUC:        0.7904


### SVM Classifier

In [74]:
from sklearn.svm import SVC
svm = SVC(kernel='rbf', random_state=42, probability=True)

In [75]:
svm.fit(X_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [76]:
y_pred = svm.predict(X_test)

In [77]:
results = evaluate_binary_classifier(svm, X_test, y_test)

=== Classification Metrics ===
Accuracy:   0.6431
F1 Score:   0.6474
Precision:  0.6441
Recall:     0.6507

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.64      0.64      0.64       288
           1       0.64      0.65      0.65       292

    accuracy                           0.64       580
   macro avg       0.64      0.64      0.64       580
weighted avg       0.64      0.64      0.64       580

=== Confusion Matrix ===
[[183 105]
 [102 190]]

=== Probability Metrics ===
Log Loss:   0.6285
Brier Score:0.2175
AUC:        0.7041


### XGBoost

In [78]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1, objective='binary:logistic')
xgb_model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [80]:
y_pred = xgb_model.predict(X_test)

In [81]:
results = evaluate_binary_classifier(xgb_model, X_test, y_test)

=== Classification Metrics ===
Accuracy:   0.7069
F1 Score:   0.7099
Precision:  0.7075
Recall:     0.7123

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.71      0.70      0.70       288
           1       0.71      0.71      0.71       292

    accuracy                           0.71       580
   macro avg       0.71      0.71      0.71       580
weighted avg       0.71      0.71      0.71       580

=== Confusion Matrix ===
[[202  86]
 [ 84 208]]

=== Probability Metrics ===
Log Loss:   0.5677
Brier Score:0.1921
AUC:        0.7741


### Ada Boost

In [92]:
from sklearn.ensemble import AdaBoostClassifier
adaboost_model = AdaBoostClassifier()
adaboost_model.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.Support for sample weighting is required, as well as proper``classes_`` and ``n_classes_`` attributes. If ``None``, thenthe base estimator is :class:`~sklearn.tree.DecisionTreeClassifier`initialized with `max_depth=1`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",50
,"learning_rate learning_rate: float, default=1.0Weight applied to each classifier at each boosting iteration. A higherlearning rate increases the contribution of each classifier. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",1.0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None


In [93]:
y_pred = adaboost_model.predict(X_test)

In [94]:
results = evaluate_binary_classifier(adaboost_model, X_test, y_test)

=== Classification Metrics ===
Accuracy:   0.6983
F1 Score:   0.6998
Precision:  0.7010
Recall:     0.6986

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.70      0.70      0.70       288
           1       0.70      0.70      0.70       292

    accuracy                           0.70       580
   macro avg       0.70      0.70      0.70       580
weighted avg       0.70      0.70      0.70       580

=== Confusion Matrix ===
[[201  87]
 [ 88 204]]

=== Probability Metrics ===
Log Loss:   0.6050
Brier Score:0.2080
AUC:        0.7667


### Voting Classifier

In [103]:
from sklearn.ensemble import VotingClassifier

voting_classifier = VotingClassifier(estimators=[("xgb", xgb_model1), ("rf", random_forest1), ("adabBoost", adaboost1)], voting="soft")

In [104]:
voting_classifier.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('xgb', ...), ('rf', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None


In [105]:
results = evaluate_binary_classifier(voting_classifier, X_test, y_test)

=== Classification Metrics ===
Accuracy:   0.6810
F1 Score:   0.6848
Precision:  0.6814
Recall:     0.6884

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.68      0.67      0.68       288
           1       0.68      0.69      0.68       292

    accuracy                           0.68       580
   macro avg       0.68      0.68      0.68       580
weighted avg       0.68      0.68      0.68       580

=== Confusion Matrix ===
[[194  94]
 [ 91 201]]

=== Probability Metrics ===
Log Loss:   0.5751
Brier Score:0.1975
AUC:        0.7658


### MLP Classifier

In [110]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(max_iter=100, random_state=42)
mlp.fit(X_train, y_train)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",100
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42


In [111]:
results = evaluate_binary_classifier(mlp, X_test, y_test)

=== Classification Metrics ===
Accuracy:   0.5069
F1 Score:   0.6713
Precision:  0.5052
Recall:     1.0000

=== Classification Report ===
              precision    recall  f1-score   support

           0       1.00      0.01      0.01       288
           1       0.51      1.00      0.67       292

    accuracy                           0.51       580
   macro avg       0.75      0.50      0.34       580
weighted avg       0.75      0.51      0.34       580

=== Confusion Matrix ===
[[  2 286]
 [  0 292]]

=== Probability Metrics ===
Log Loss:   1.5744
Brier Score:0.4211
AUC:        0.7790
